# Задача 1: Лингвистика и синтаксический разбор

## Импорт необходимых библиотек

In [43]:
# Стандартные библиотеки
from collections import Counter
import sqlite3
import re

# Библиотеки для визуализации
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

# Лингвистические библиотеки
import spacy

from pymorphy3 import MorphAnalyzer

## Загрузка данных

In [37]:
def load_texts(file_path):
    """
    Загружает текстовые данные из БД.
    
    Args:
        file_path: путь к БД с новостными статьями
    
    Returns:
        df: DataFrame с одним столбцом текстов статей (он уже очищен)
    """
    with sqlite3.connect(file_path) as conn:
        query = """
        SELECT description as text FROM articles WHERE description IS NOT NULL
        """

        df = pd.read_sql(query, conn)
    return df


In [39]:
df = load_texts('data/lenta.db')
df.head(10)


,text
0,"Microsoftопровергла сведения, что она собирает..."
1,"Астрономы предложили гипотезу, описывающую эво..."
2,Десятилетний мальчик (его имя не называется) п...
3,"Песня группы The Beatles ""Hey Jude"" на данный ..."
4,IPO (первичное размещение акций на бирже) Пром...
5,В Санкт-Петербурге были избиты несколько гражд...
6,Компания Coca-Cola в 12-й раз подряд заняла пе...
7,"Кандидат на пост мэра Москвы, глава аппарата п..."
8,"Ведущий грузинского телеканала ""Маэстро"" Васик..."
9,Премьер-министрВладимир Путиннашел две амфоры ...


## Инициализация инструментов

In [31]:
# Инициализация pymorphy3
morph = MorphAnalyzer()

# Инициализация spacy
nlp = spacy.load("ru_core_news_md")

In [42]:
def clean_text(text):
    """
    Полная очистка текста с обработкой скобок и знаков препинания.
    """
    if not isinstance(text, str):
        return ""

    # 1. Замена спецсимволов переноса на пробелы
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')

    # 2. Разделение стыков Латиница/Кириллица
    text = re.sub(r'([a-zA-Z])([а-яА-Я])', r'\1 \2', text)
    text = re.sub(r'([а-яА-Я])([a-zA-Z])', r'\1 \2', text)

    # 3. Обработка ЗНАКОВ ПРЕПИНАНИЯ
    punctuation = ',.:;!?'

    # 3а. Убираем пробелы ПЕРЕД этими знаками ("слово ,") -> "слово,"
    pattern_before = r'\s+([' + re.escape(punctuation) + r'])'
    text = re.sub(pattern_before, r'\1', text)

    # 3б. Добавляем пробел ПОСЛЕ этих знаков, если там сразу буква ("слово,слово") -> "слово, слово"
    pattern_after = r'([' + re.escape(punctuation) + r'])([А-Яа-яA-Za-z])'
    text = re.sub(pattern_after, r'\1 \2', text)

    # 4. Обработка СКОБОК ()
    # 4а. Открывающая скобка: убираем пробел ВНУТРИ после неё "( слово)" -> "(слово)"
    text = re.sub(r'\(\s+', r'(', text)
    # 4б. Открывающая скобка: добавляем пробел ПЕРЕД ней, если там буква "слово(" -> "слово ("
    text = re.sub(r'([А-Яа-яA-Za-z0-9])\(', r'\1 (', text)

    # 4в. Закрывающая скобка: убираем пробел ВНУТРИ перед ней "(слово )" -> "(слово)"
    text = re.sub(r'\s+\)', r')', text)
    # 4г. Закрывающая скобка: добавляем пробел ПОСЛЕ неё, если там буква ")слово" -> ") слово"
    text = re.sub(r'\)([А-Яа-яA-Za-z0-9])', r') \1', text)

    # 5. все множественные пробелы превращаем в один
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [44]:
# Чисти-чисти
print('Чистим данные')
df['clean_text'] = df['text'].progress_apply(clean_text)
df['clean_text'].head()

Чистим данные


100%|██████████| 99999/99999 [00:26<00:00, 3807.60it/s]


0    Microsoft опровергла сведения, что она собирае...
1    Астрономы предложили гипотезу, описывающую эво...
2    Десятилетний мальчик (его имя не называется) п...
3    Песня группы The Beatles "Hey Jude" на данный ...
4    IPO (первичное размещение акций на бирже) Пром...
Name: clean_text, dtype: str

In [45]:
def extract_pairs(text):
    """Генератор, который выдает все пары (подлежащее, сказуемое) в предложении"""
    doc = nlp(text)
    found_pairs = []
    for sent in doc.sents:
        predicates = []

        for token in sent:
            # Проверка глаголов и вспомогательных глаголов
            if token.pos_ in ['VERB', 'AUX']:
                # Поиска среди найденных сказуемого
                if token.dep_ == "ROOT":
                    predicates.append(token)
                elif token.dep_ == "conj":
                    predicates.append(token)

        # Для каждого сказуемого ищем его подлежащее
        for pred in predicates:
            subject = None
            # Ищем среди детей сказуемого
            for child in pred.children:
                if child.dep_ == "nsubj":
                    subject = child
                    break

            # Если пара найдена, сохраняем леммы в нижнем регистре
            if subject and subject.pos_ in ["NOUN", "PROPN", "PRON"]: # Подлежащее обычно существительное, имя собственное или местоимение
                found_pairs.append((subject.lemma_.lower(), pred.lemma_.lower()))

    return found_pairs

In [47]:
# прогресс-бар для pandas
tqdm.pandas()

# Сначала протестируем на 100 строках
print("Чтобы было видно progress-bar")
test_result = df['clean_text'].head(100).progress_apply(extract_pairs)

test_result.head()


Чтобы было видно progress-bar


100%|██████████| 100/100 [00:06<00:00, 15.26it/s]


0    [(microsoft, опровергнуть), (представитель, за...
1    [(астроном, предложить), (учёный, находить), (...
2    [(мальчик, позвонить), (инцидент, произойти), ...
3    [(песня, являться), (композиция, занимать), (п...
4    [(президент, заявить), (приём, начинать), (он,...
Name: clean_text, dtype: object

In [34]:
print("\nЗапуск обработки всего датасета...")
df['pairs'] = df['text'].progress_apply(extract_pairs)


Запуск обработки всего датасета...


  1%|          | 1171/99999 [01:16<1:48:14, 15.22it/s]


KeyboardInterrupt: 

## Синтаксический разбор предложений

In [ ]:
def parse_sentence(sentence, segmenter, morph_tagger, syntax_parser):
    """
    Выполняет синтаксический разбор предложения и выделяет подлежащее и сказуемое.
    
    Args:
        sentence: строка с предложением
        segmenter: объект Segmenter из natasha
        morph_tagger: объект MorphTagger из natasha
        syntax_parser: объект SyntaxParser из natasha
    
    Returns:
        tuple: (подлежащее, сказуемое) или (None, None) если не найдено
    """
    # TODO: реализовать синтаксический разбор с помощью natasha
    # Подсказка:
    # 1. Создать объект Doc: doc = Doc(sentence)
    # 2. Применить segmenter: doc.segment(segmenter)
    # 3. Применить morph_tagger: doc.tag_morph(morph_tagger)
    # 4. Применить syntax_parser: doc.parse_syntax(syntax_parser)
    # 5. Найти подлежащее (токен с зависимостью 'nsubj' или 'nsubj:pass')
    #    Пример: для токена token, проверить token.rel == 'nsubj'
    # 6. Найти сказуемое (токен с зависимостью 'root' или связанный с подлежащим)
    #    Пример: для токена token, проверить token.rel == 'root'
    # Доступ к токенам: doc.sents[0].tokens
    # Доступ к тексту токена: token.text
    # Доступ к синтаксическим зависимостям: token.rel для отношения, token.head для головы
    ...


In [ ]:
def build_cooccurrence_dependencies(texts, segmenter, morph_tagger, syntax_parser):
    """
    Проверяет, является ли слово сказуемым.
    
    Args:
        word: слово для проверки
        morph: объект MorphAnalyzer из pymorphy3
    
    Returns:
        bool: True если слово является сказуемым
    """
    # TODO: реализовать проверку с помощью pymorphy3
    # Подсказка:
    # 1. Использовать morph.parse(word) для получения морфологического разбора
    # 2. Проверить, что часть речи - сказуемое
    # 3. Проверить, что слово может быть сказуемым (например, в краткой форме)
    pass


In [ ]:
def build_cooccurrence_dependencies(texts, segmenter, morph_tagger, syntax_parser):
    """
    Строит зависимости совместных употреблений подлежащих и сказуемых.
    
    Args:
        texts: список предложений
        segmenter, morph_tagger, syntax_parser: объекты из natasha
        morph: объект MorphAnalyzer из pymorphy3
    
    Returns:
        Counter: счетчик пар (подлежащее, сказуемое)
    """
    # TODO: реализовать построение зависимостей
    # Подсказка:
    # 1. Для каждого предложения вызвать parse_sentence
    # 2. Если найдены подлежащее и сказуемое, добавить пару (подлежащее, сказуемое) в список
    # 3. Использовать collections.Counter для подсчета частот
    # Пример: counter = Counter([('студент', 'учится'), ('преподаватель', 'объясняет'), ...])
    
    cooccurrences = []
    
    # Ваш код здесь
    # for sentence in texts:
    #     subject, predicate = parse_sentence(sentence, segmenter, morph_tagger, syntax_parser)
    #     if subject and predicate:
    #         cooccurrences.append((subject, predicate))
    
    return Counter(cooccurrences)


## Визуализация результатов

In [ ]:
def visualize_results(counter, top_n=20):
    """
    Визуализирует результаты анализа.
    
    Args:
        counter: Counter с парами (подлежащее, сказуемое)
        top_n: количество топовых сочетаний для отображения
    """
    # TODO: реализовать визуализацию
    # Подсказка:
    # 1. Получить top_n наиболее частых сочетаний
    # 2. Построить график (столбчатая диаграмма, например)
    # 3. Можно использовать matplotlib / seaborn / pandas / любое другое решенеие для визуализации
    ...


## Основной код выполнения задачи

In [ ]:
# Загрузка данных
# texts = load_texts('./data/texts.txt')
# print(f"Загружено предложений: {len(texts)}")

# Построение зависимостей
# cooccurrences = build_cooccurrence_dependencies(
#     texts, 
#     segmenter, 
#     morph_tagger, 
#     syntax_parser, 
#     morph
# )

# Вывод результатов
# print("\nТоп-10 наиболее частых сочетаний:")
# for (subject, predicate), count in cooccurrences.most_common(10):
#     print(f"{subject} - {predicate}: {count}")

# Визуализация
# visualize_results(cooccurrences, top_n=20)
